In [1]:
from pyspark.sql import SparkSession

In [2]:
# Création de la session Spark avec S3 ET Postgres
spark = SparkSession.builder \
    .appName("NYC_Taxi_Processing") \
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-aws:3.3.4," +
            "com.amazonaws:aws-java-sdk-bundle:1.12.262," +
            "org.postgresql:postgresql:42.7.2") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

:: loading settings :: url = jar:file:/home/airflow/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/airflow/.ivy2/cache
The jars for the packages stored in: /home/airflow/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a7162d5c-b26c-4acb-b37c-1dea2a8a122c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (118ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] co

In [3]:
df = spark.read.parquet("s3a://nyc-taxi/trajets/yellow_tripdata_2026-01.parquet", header=True, inferSchema=True)


26/04/30 13:16:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
#pour un meilleur affichage
df.show(5, vertical=True, truncate=False)


-RECORD 0------------------------------------
 VendorID              | 2                   
 tpep_pickup_datetime  | 2026-01-01 00:54:04 
 tpep_dropoff_datetime | 2026-01-01 00:59:37 
 passenger_count       | 1                   
 trip_distance         | 0.97                
 RatecodeID            | 1                   
 store_and_fwd_flag    | N                   
 PULocationID          | 239                 
 DOLocationID          | 238                 
 payment_type          | 1                   
 fare_amount           | 7.2                 
 extra                 | 1.0                 
 mta_tax               | 0.5                 
 tip_amount            | 3.66                
 tolls_amount          | 0.0                 
 improvement_surcharge | 1.0                 
 total_amount          | 15.86               
 congestion_surcharge  | 2.5                 
 Airport_fee           | 0.0                 
 cbd_congestion_fee    | 0.0                 
-RECORD 1-------------------------

On va afficher le nomle nombre de cellules vides par colonne

In [5]:
from pyspark.sql import functions as F

# On compte uniquement les NULL (marche pour les dates, les nombres et le texte)
df_empty_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df.columns
])

# Affichage horizontal lisible
df_empty_counts.toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,0,0,0,1088058,0,1088058,1088058,0,0,0,0,0,0,0,0,0,0,1088058,1088058,0


En visualisant ce résultat, nous voyons que certains trajets n'ont donc pas de passager_count défini. On va donc faire le traitement des calculs de temps de trajets... en partant du principe qu'il y a au moins un passager vu que (tpep_pickup_datetime) et (tpep_dropoff_datetime) ne contiennent pas de valeures nulles

In [6]:
#Maintenant on calcul la durée de trajet pour chaque colonne
duree_trajet = (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60

df.select(
    "tpep_pickup_datetime", 
    "tpep_dropoff_datetime", 
    duree_trajet.alias("durée trajet (minutes)") 
).show(5)

+--------------------+---------------------+----------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|durée trajet (minutes)|
+--------------------+---------------------+----------------------+
| 2026-01-01 00:54:04|  2026-01-01 00:59:37|                  5.55|
| 2026-01-01 00:34:04|  2026-01-01 00:39:47|     5.716666666666667|
| 2026-01-01 00:57:06|  2026-01-01 01:05:59|     8.883333333333333|
| 2026-01-01 00:15:22|  2026-01-01 00:58:10|                  42.8|
| 2026-01-01 00:27:13|  2026-01-01 00:40:43|                  13.5|
+--------------------+---------------------+----------------------+
only showing top 5 rows



On voit bien que la colonne (durée) affiche bien la durée des trajets en minutes. On passe ensuite aux autres colonnes.

On va maintenant catégoriser les distances parcourues en catégories

In [7]:
#on convertit les miles en kilometres
distance_parcourue = F.col("trip_distance") * 1.609

#on attribue les catégories
categorie_distance = (F.when(distance_parcourue <= 2, "0-2 km")
                  .when((distance_parcourue > 2) & (distance_parcourue <= 5), "2-5 km")
                  .otherwise(">5 km"))

df.select(
    "trip_distance", 
    categorie_distance.alias("categorie distance") 
).show(5)

+-------------+------------------+
|trip_distance|categorie distance|
+-------------+------------------+
|         0.97|            0-2 km|
|          0.9|            0-2 km|
|          1.4|            2-5 km|
|         5.58|             >5 km|
|         2.16|            2-5 km|
+-------------+------------------+
only showing top 5 rows



On voit que le résultat est bon. Passons maintenant au Type de paiement (via table de correspondance). Pour ce faire on regarde le Data Dictionnary sur le site.

In [8]:
# On fait la correspondance
type_paiement = (F.when(F.col("payment_type") == 0, "Flex Fare trip")
                       .when(F.col("payment_type") == 1, "Credit card")
                       .when(F.col("payment_type") == 2, "Cash")
                       .when(F.col("payment_type") == 3, "No charge")
                       .when(F.col("payment_type") == 4, "Dispute")
                       .when(F.col("payment_type") == 5, "Unknown")
                       .when(F.col("payment_type") == 6, "Voided trip")
                       .otherwise("Other/Unknown"))

df.select(
    "payment_type",
    type_paiement.alias("type de paiement")
).show(10)

+------------+----------------+
|payment_type|type de paiement|
+------------+----------------+
|           1|     Credit card|
|           2|            Cash|
|           1|     Credit card|
|           1|     Credit card|
|           1|     Credit card|
|           1|     Credit card|
|           2|            Cash|
|           1|     Credit card|
|           1|     Credit card|
|           1|     Credit card|
+------------+----------------+
only showing top 10 rows



On voit que cela a bien fonctionné.On passe ensuite au calcul du pourcentage de pourboire

In [9]:
# Calcul du pourcentage de pourboire en verifiant que fareamount different de 0
pourcentage_pourboire = (F.when(F.col("fare_amount") > 0, 
                               (F.col("tip_amount") / F.col("fare_amount")) * 100)
                         .otherwise(0))

df.select(
    "fare_amount", 
    "tip_amount", 
    pourcentage_pourboire.alias("pourcentage pourboire")
).show(10)

+-----------+----------+---------------------+
|fare_amount|tip_amount|pourcentage pourboire|
+-----------+----------+---------------------+
|        7.2|      3.66|    50.83333333333333|
|        7.9|       0.0|                  0.0|
|       10.7|       2.5|   23.364485981308412|
|       38.7|     11.11|    28.70801033591731|
|       13.5|      3.85|    28.51851851851852|
|       14.2|      4.99|   35.140845070422536|
|       11.4|       0.0|                  0.0|
|       22.6|      5.65|                 25.0|
|       37.3|      8.61|   23.083109919571047|
|       10.7|      2.36|    22.05607476635514|
+-----------+----------+---------------------+
only showing top 10 rows



On voit que cela a bien fonctionné.On passe ensuite au calcul de  Heure de prise en charge, jour de la semaine

In [10]:
# On extrait l'heure de la date
heure_prisencharge = F.hour("tpep_pickup_datetime")

# On extrait le jour
jour_prisencharge = F.dayofweek("tpep_pickup_datetime")

# On convertit les numeros jour en texte
jour_prisencharge = (F.when(jour_prisencharge == 1, "Dimanche")
                  .when(jour_prisencharge == 2, "Lundi")
                  .when(jour_prisencharge == 3, "Mardi")
                  .when(jour_prisencharge == 4, "Mercredi")
                  .when(jour_prisencharge == 5, "Jeudi")
                  .when(jour_prisencharge == 6, "Vendredi")
                  .when(jour_prisencharge == 7, "Samedi"))
df.select(
    "tpep_pickup_datetime",
    heure_prisencharge.alias("heure prise en charge"),
    jour_prisencharge.alias("jour prise en charge")
).show(10)

+--------------------+---------------------+--------------------+
|tpep_pickup_datetime|heure prise en charge|jour prise en charge|
+--------------------+---------------------+--------------------+
| 2026-01-01 00:54:04|                    0|               Jeudi|
| 2026-01-01 00:34:04|                    0|               Jeudi|
| 2026-01-01 00:57:06|                    0|               Jeudi|
| 2026-01-01 00:15:22|                    0|               Jeudi|
| 2026-01-01 00:27:13|                    0|               Jeudi|
| 2026-01-01 00:47:11|                    0|               Jeudi|
| 2026-01-01 00:17:54|                    0|               Jeudi|
| 2026-01-01 00:34:28|                    0|               Jeudi|
| 2026-01-01 00:34:14|                    0|               Jeudi|
| 2026-01-01 00:41:07|                    0|               Jeudi|
+--------------------+---------------------+--------------------+
only showing top 10 rows



On voit que cela a bien fonctionné. On passe ensuite au calcul de Informations de zone (via pickup_location_id et table des zones si dispo).
On va le faire en téléchargeant la table de mapping des zones sur le sites

In [11]:
# on le récuprère puis on le lit
df_zones = spark.read.csv("s3a://nyc-taxi/trajets/taxi_zone_lookup.csv", header=True, inferSchema=True)

# on affiche voir
df_zones.show(5)


+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



Maintenant on va faire la jointure de cette table avec notre Dataframe initial 
ainsi que la jointure avec toutes les colonnes précédemment faites


In [12]:
df_zones = df_zones.withColumn("LocationID", F.col("LocationID").cast("int"))

df_jointure_zones = df.join(
    df_zones, 
    df.PULocationID == df_zones.LocationID, 
    how="left"
).drop("LocationID") # évier le doublon de colonnes

df_jointure_zones.show(5, vertical=True, truncate=False)

-RECORD 0-----------------------------------------
 VendorID              | 2                        
 tpep_pickup_datetime  | 2026-01-01 00:54:04      
 tpep_dropoff_datetime | 2026-01-01 00:59:37      
 passenger_count       | 1                        
 trip_distance         | 0.97                     
 RatecodeID            | 1                        
 store_and_fwd_flag    | N                        
 PULocationID          | 239                      
 DOLocationID          | 238                      
 payment_type          | 1                        
 fare_amount           | 7.2                      
 extra                 | 1.0                      
 mta_tax               | 0.5                      
 tip_amount            | 3.66                     
 tolls_amount          | 0.0                      
 improvement_surcharge | 1.0                      
 total_amount          | 15.86                    
 congestion_surcharge  | 2.5                      
 Airport_fee           | 0.0   

on rajoute toutes les autres colonnes précédentes

In [13]:
# L'ouverture de la parenthèse avant "df" permet de sauter des lignes librement
df_final = (df_jointure_zones.withColumn("categorie_distance", categorie_distance)
              .withColumn("duree_trajet", duree_trajet) 
              .withColumn("pourcentage_pourboire", pourcentage_pourboire) 
              .withColumn("heure_prisencharge", heure_prisencharge) 
              .withColumn("jour_prisencharge", jour_prisencharge)
              .withColumn("type_paiement", type_paiement))


# Affiche le premier record en entier
df_final.show(1, vertical=True)

26/04/30 13:16:38 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0-------------------------------------
 VendorID              | 2                    
 tpep_pickup_datetime  | 2026-01-01 00:54:04  
 tpep_dropoff_datetime | 2026-01-01 00:59:37  
 passenger_count       | 1                    
 trip_distance         | 0.97                 
 RatecodeID            | 1                    
 store_and_fwd_flag    | N                    
 PULocationID          | 239                  
 DOLocationID          | 238                  
 payment_type          | 1                    
 fare_amount           | 7.2                  
 extra                 | 1.0                  
 mta_tax               | 0.5                  
 tip_amount            | 3.66                 
 tolls_amount          | 0.0                  
 improvement_surcharge | 1.0                  
 total_amount          | 15.86                
 congestion_surcharge  | 2.5                  
 Airport_fee           | 0.0                  
 cbd_congestion_fee    | 0.0                  
 Borough     

In [14]:
#MAINTENANT QUE NOS TESTS DE TRANSFORMATIONS ONT ETE FAITS ONT VA CREE NOS SCRIPTS PY DE TRANSFORMATIONS ET Sauvegarder le résultat dans PostgreSQL sous le nom fact_taxi_trips